# Grover's Algorithm Notebook
Examples in Qiskit, PennyLane, and TensorFlow Quantum (Cirq-based).

## Install (if needed)
```bash
pip install qiskit pennylane tensorflow-quantum cirq
```

## Qiskit Example

In [ ]:

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit import transpile

qc = QuantumCircuit(2,2)
# superposition
qc.h([0,1])
# oracle marks |11>
qc.cz(0,1)
# diffuser
qc.h([0,1]); qc.x([0,1]); qc.h(1); qc.cx(0,1); qc.h(1); qc.x([0,1]); qc.h([0,1])
qc.measure([0,1],[0,1])

sim = AerSimulator()
tqc = transpile(qc, sim)
result = sim.run(tqc, shots=1024).result()
print(result.get_counts())
qc.draw('text')


## PennyLane Example

In [ ]:

import pennylane as qml
from pennylane import numpy as np

dev = qml.device("default.qubit", wires=2, shots=1000)

@qml.qnode(dev)
def grover():
    qml.Hadamard(0); qml.Hadamard(1)
    # oracle |11>
    qml.CZ(wires=[0,1])
    # diffuser
    for w in [0,1]:
        qml.Hadamard(w); qml.PauliX(w)
    qml.Hadamard(1); qml.CNOT(wires=[0,1]); qml.Hadamard(1)
    for w in [0,1]:
        qml.PauliX(w); qml.Hadamard(w)
    return qml.sample()

samples = grover()
# convert samples to bitstrings
counts={}
for s in samples:
    b=''.join(str(int(x)) for x in s)
    counts[b]=counts.get(b,0)+1
print(counts)


## TensorFlow Quantum Example

In [ ]:

import tensorflow as tf
import tensorflow_quantum as tfq
import cirq, sympy

q0,q1 = cirq.GridQubit.rect(1,2)

circuit = cirq.Circuit()
circuit.append([cirq.H(q0), cirq.H(q1)])
# oracle |11>
circuit.append(cirq.CZ(q0,q1))
# diffuser
circuit.append([cirq.H(q0), cirq.H(q1), cirq.X(q0), cirq.X(q1)])
circuit.append([cirq.H(q1), cirq.CNOT(q0,q1), cirq.H(q1)])
circuit.append([cirq.X(q0), cirq.X(q1), cirq.H(q0), cirq.H(q1)])

tensor = tfq.convert_to_tensor([circuit])
readouts = [cirq.Z(q0), cirq.Z(q1)]
layer = tfq.layers.Sample()
samples = layer(tensor, repetitions=1000)
print(samples)
print(circuit)


## Notes
For 2 qubits with one marked state, a single Grover iteration is optimal.